<a href="https://colab.research.google.com/github/himBabbar/chatbot/blob/main/Ollama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyngrok ollama

In [2]:
import IPython
import subprocess
from pyngrok import ngrok
from google.colab import userdata
from typing import Any, Dict, List, Optional

In [3]:
def start_ollama_server() -> None:
    """Starts the Ollama server."""
    subprocess.Popen(['ollama', 'serve'])
    print("Ollama server started.")


def check_ollama_port(port: str) -> None:
    """Check if Ollama server is running at the specified port."""
    try:
        subprocess.run(['sudo', 'lsof', '-i', '-P', '-n'], check=True, capture_output=True, text=True)
        if any(f":{port} (LISTEN)" in line for line in subprocess.run(['sudo', 'lsof', '-i', '-P', '-n'], capture_output=True, text=True).stdout.splitlines()):
            print(f"Ollama is listening on port {port}")
        else:
            print(f"Ollama does not appear to be listening on port {port}.")
    except subprocess.CalledProcessError as e:
         print(f"Error checking Ollama port: {e}")


def setup_ngrok_tunnel(port: str) -> ngrok.NgrokTunnel:
    """Sets up an ngrok tunnel.

    Args:
        port: The port to tunnel.

    Returns:
        The ngrok tunnel object.

    Raises:
        RuntimeError: If the ngrok authtoken is not set.
    """
    ngrok_auth_token = userdata.get('NGROK_AUTHTOKEN')
    if not ngrok_auth_token:
        raise RuntimeError("NGROK_AUTHTOKEN is not set.")

    ngrok.set_auth_token(ngrok_auth_token)
    tunnel = ngrok.connect(port, host_header=f'localhost:{port}')
    print(f"ngrok tunnel created: {tunnel.public_url}")
    return tunnel

In [4]:
NGROK_PORT = '11434'

In [5]:
# Define the port for Streamlit
STREAMLIT_PORT = '8501'

In [6]:
# Install zstd if not already installed, as required by Ollama
!sudo apt-get install -y zstd

# Install Ollama (this will re-run the installation if needed)
!curl -fsSL https://ollama.com/install.sh | sh
# Start the Ollama server
start_ollama_server()

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (500 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently i

In [22]:
check_ollama_port(NGROK_PORT)

Ollama is listening on port 11434


In [8]:
ngrok_tunnel = setup_ngrok_tunnel(NGROK_PORT)

ngrok tunnel created: https://crimp-dribble-rind.ngrok-free.dev


In [9]:
MODEL_ID = 'gemma3:4b'
PROMPT = 'Why is the sky blue?'

In [10]:
print(f'export OLLAMA_HOST={ngrok_tunnel.public_url}')

export OLLAMA_HOST=https://crimp-dribble-rind.ngrok-free.dev


In [11]:
# Set environment variables for the Streamlit app to pick up
import os
os.environ["OLLAMA_HOST"] = ngrok_tunnel.public_url
os.environ["MODEL_ID"] = MODEL_ID

In [12]:
import os
print(f"OLLAMA_HOST from os.environ: {os.environ.get('OLLAMA_HOST')}")

OLLAMA_HOST from os.environ: https://crimp-dribble-rind.ngrok-free.dev


In [13]:
print(f'ollama run {MODEL_ID}')

ollama run gemma3:4b


In [14]:
!pip install ollama

In [15]:
import ollama

In [16]:
# Install Streamlit library
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 81.8 MB/s eta 0:00:00


In [17]:
import ollama

In [18]:
def query_ollama_with_client(prompt: str, model_id: str, img_path: list = None) -> None:
    """Queries the Ollama server using the `ollama-python` client library.

    Args:
        prompt: The prompt to send to the model.
        model_id: The ID of the model to use.
    """
    try:
        messages=[
            {
                'role': 'user',
                'content': prompt,
            },
        ]

        if img_path:
          messages[0]['images'] = [img_path]

        response = client.chat(
            model=model_id,
            messages=messages
        )
        print("Response from ollama client:")
        print(response['message']['content'])
    except Exception as e:
        print(f"Error querying Ollama with client: {e}")

In [19]:
# Setup an Ollama client with the provided host
client = ollama.Client(host=ngrok_tunnel.public_url)

In [20]:
# Load the model at the client
client.pull(model=MODEL_ID)

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [21]:
# The 'nvidia-smi' command is unrelated to building a chat UI.
# I'll add the chat UI in a new cell for better organization.
# This cell can be kept as is, or removed if not needed.